# Total Snippets Extraction
This notebook extract all opium mentions from the full library to prepare opium snippets for topic modelling (see 3_analysis).

In [21]:
import os
import re
import pandas as pd
import spacy
import ipywidgets as widgets
from tqdm.auto import tqdm

from IPython.display import display, HTML

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import spacy.cli
    spacy.cli.download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

In [12]:
# Setup Paths
metadata_path = '../../data/metadata_files/GP_opium_filtered_1850_1930.parquet'
fulltext_path = '../../opium_books_fulltext'

output_parquet = '../../data/snippets/total_snippets.parquet'
output_csv = '../../data/snippets/total_snippets.csv'

window_size = 100

## 1. Extracting context windows out of all books

In [24]:
books = pd.read_parquet(metadata_path)
books["Etext Number"]

0          15
1          16
3          27
6          44
7          60
        ...  
6396    74956
6403    75246
6406    75372
6407    75497
6408    75518
Name: Etext Number, Length: 4378, dtype: int64

In [28]:
keywords = books['Opium Keywords'].explode().unique().tolist()
keywords

['opium',
 'paregoric',
 'laudanum',
 'heroin',
 'narcotic',
 'morphine',
 'anodyne',
 'soporific',
 'nepenthe',
 'chandu',
 'codein',
 'dover’s powder']

In [36]:
results = []
if os.path.exists(fulltext_path):
    for book_id in book_ids:
        filepath = os.path.join(fulltext_path, str(book_id))
        if os.path.isfile(filepath):
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
                text_lower = text.lower()
                for keyword in keywords:
                    pattern = r'\b' + re.escape(keyword) + r'\b'
                    for match in re.finditer(pattern, text_lower):
                        idx = match.start()
                        left_text = text[:idx]
                        right_text = text[idx + len(keyword):]
                        
                        left_words = left_text.split()
                        right_words = right_text.split()
                        
                        context_left = ' '.join(left_words[-window_size:])
                        context_right = ' '.join(right_words[:window_size])
                        
                        results.append({
                            'Book_ID': book_id,
                            'Keyword': keyword,
                            'Snippet': f"{context_left} {text[idx:idx+len(keyword)]} {context_right}"
                        })
                        
df = pd.DataFrame(results)
print(f"Found {len(df)} keyword mentions across all {len(book_ids)} books.")


Found 6658 keyword mentions across all 4378 books.


Also we remove overlapping snippets in 2 ways:
1. Within-book: if two snippets from same book + same keyword are too similar OR one contains the other → drop one
2. Cross-book: detect edition duplicates (text similarity, high threshold) → drop one

In [43]:
def normalize(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def deduplicate_snippets(df,
                         within_book_sim=0.85,
                         cross_book_sim=0.95):

    print(f"Original snippets: {len(df)}")

    df = df.copy()
    df['norm'] = df['Snippet'].map(normalize)

    to_drop = set()

    # ---------------------------
    # 1. WITHIN-BOOK DEDUP
    # ---------------------------
    for book_id, group in df.groupby('Book_ID'):

        idxs = group.index.tolist()

        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):

                a = idxs[i]
                b = idxs[j]

                if a in to_drop or b in to_drop:
                    continue

                s1 = df.at[a, 'norm']
                s2 = df.at[b, 'norm']

                # containment-aware similarity (good for overlapping windows)
                sim = fuzz.token_set_ratio(s1, s2) / 100

                if sim >= within_book_sim:
                    # drop shorter / redundant one
                    if len(s1) <= len(s2):
                        to_drop.add(a)
                    else:
                        to_drop.add(b)

    # ---------------------------
    # 2. CROSS-BOOK DEDUP (EDITION MATCHING)
    # ---------------------------
    remaining = df.drop(index=list(to_drop)).copy()
    remaining_idxs = remaining.index.tolist()

    for i in range(len(remaining_idxs)):
        for j in range(i + 1, len(remaining_idxs)):

            a = remaining_idxs[i]
            b = remaining_idxs[j]

            if a in to_drop or b in to_drop:
                continue

            # skip same book here (already handled above)
            if remaining.at[a, 'Book_ID'] == remaining.at[b, 'Book_ID']:
                continue

            s1 = remaining.at[a, 'norm']
            s2 = remaining.at[b, 'norm']

            sim = fuzz.token_set_ratio(s1, s2) / 100

            if sim >= cross_book_sim:
                # treat as duplicate edition snippet
                to_drop.add(b)

    # final output
    df_dedup = df.drop(index=list(to_drop)).drop(columns=['norm'])

    print(f"Dropped {len(to_drop)} snippets")
    print(f"Remaining snippets: {len(df_dedup)}")

    return df_dedup

In [44]:
df_dedup = deduplicate_snippets(df)

Original snippets: 6658
Dropped 1288 snippets
Remaining snippets: 5370


In [46]:
df_dedup[['Book_ID', 'Keyword', 'Snippet']].to_csv(output_csv)
df_dedup[['Book_ID', 'Keyword', 'Snippet']].to_parquet(output_parquet)

## 2. Manual Inspection Viewer
Use the slider to browse through the extracted snippets. The target keyword is highlighted.

In [ ]:
pd.set_option('display.max_colwidth', None)
def view_snippet(index):
    if len(df_dedup) == 0:
        print("No snippets found.")
        return
    row = df_dedup.iloc[index]
    html_out = f"""
    <div style='font-family: Georgia, serif; font-size: 16px; line-height: 1.6; max-width: 800px; padding: 20px; border: 1px solid #ccc; border-radius: 5px; background: #f9f9f9;'>
        <h4>Book ID: {row['Book_ID']} | Keyword: <span style='color: dimgrey;'>{row['Keyword'].upper()}</span></h4>
        <hr>
        <p>
            {row['Left_Context']} 
            <span style='background-color: #ffeb3b; font-weight: bold; padding: 0 4px;'>{row['Keyword']}</span> 
            {row['Right_Context']}
        </p>
    </div>
    """
    display(HTML(html_out))

In [ ]:
if len(df) > 0:
    slider = widgets.IntSlider(min=0, max=len(df_dedup)-1, step=1, description='Snippet:', layout=widgets.Layout(width='800px'))
    widgets.interact(view_snippet, index=slider)

interactive(children=(IntSlider(value=0, description='Snippet:', layout=Layout(width='800px'), max=3856), Outp…